# Explore shufinskiy `nbastats` Data

Goal: inspect one local shufinskiy season before designing the M1 schema.

The key question is whether this file is player-game box scores or raw play-by-play events.

## Setup

Use the already-downloaded CSV so this notebook is deterministic and does not hit `stats.nba.com`.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

csv_path = Path("nbastats_2024.csv")
if not csv_path.exists():
    csv_path = Path("notebooks/nbastats_2024.csv")

csv_path.exists(), csv_path

In [ ]:
# Keep IDs as strings and keep blanks as blanks so participant slots are easy to inspect.
pbp = pd.read_csv(csv_path, dtype=str, keep_default_na=False)
pbp.shape

## Raw Columns And Sample Rows

In [ ]:
pbp.columns.tolist()

In [ ]:
pbp.head(10)

## Is This Player-Game Data?

A player-game dataset should have one row per player per game and direct stat columns like `PTS`, `REB`, `AST`, and `MIN`.

This file has one row per event.

In [ ]:
summary = {
    "rows": len(pbp),
    "columns": pbp.shape[1],
    "unique_games": pbp["GAME_ID"].nunique(),
    "rows_per_game_avg": len(pbp) / pbp["GAME_ID"].nunique(),
}
summary

In [ ]:
expected_player_log_cols = ["PTS", "REB", "AST", "MIN", "GAME_DATE"]
missing_player_log_cols = [col for col in expected_player_log_cols if col not in pbp.columns]
missing_player_log_cols

Conclusion: this is not the direct source for `player_game_logs`. Use `nba_api` for box scores/player game logs. Use shufinskiy for raw play-by-play.

## Game Identity

`GAME_ID` identifies the game. `EVENTNUM` identifies the event within that game. This pair is a candidate natural key, but the raw file should still be checked for duplicates before enforcing a unique constraint.

In [ ]:
pbp[["GAME_ID", "EVENTNUM", "EVENTMSGTYPE", "PERIOD", "PCTIMESTRING", "HOMEDESCRIPTION", "VISITORDESCRIPTION", "NEUTRALDESCRIPTION"]].head(20)

In [ ]:
# Candidate natural key check for a raw play-by-play table.
pbp.duplicated(["GAME_ID", "EVENTNUM"]).sum()

In [ ]:
duplicate_events = pbp[pbp.duplicated(["GAME_ID", "EVENTNUM"], keep=False)].sort_values(["GAME_ID", "EVENTNUM"])
duplicate_events[["GAME_ID", "EVENTNUM", "EVENTMSGTYPE", "PERIOD", "PCTIMESTRING", "HOMEDESCRIPTION", "VISITORDESCRIPTION", "SCORE", "SCOREMARGIN"]]

## Date And Time Fields

There is no `GAME_DATE` column here. The file has wall-clock and period-clock strings.

In [ ]:
time_cols = [col for col in pbp.columns if "DATE" in col.upper() or "TIME" in col.upper() or "CLOCK" in col.upper()]
time_cols

In [ ]:
pbp[["GAME_ID", "PERIOD", "WCTIMESTRING", "PCTIMESTRING", "NEUTRALDESCRIPTION"]].head(20)

Schema implication: shufinskiy alone is not enough to build exact `scheduled_tip_time_utc`. M1 game metadata needs `nba_api` schedule/box-score data or another source with game dates.

## Team Identity

Team identity is attached to player participant slots, not to one row-level `TEAM_ID` column.

In [ ]:
team_cols = [col for col in pbp.columns if "TEAM" in col]
team_cols

In [ ]:
team_frames = []
for slot in ["PLAYER1", "PLAYER2", "PLAYER3"]:
    cols = [f"{slot}_TEAM_ID", f"{slot}_TEAM_CITY", f"{slot}_TEAM_NICKNAME", f"{slot}_TEAM_ABBREVIATION"]
    frame = pbp.loc[pbp[f"{slot}_TEAM_ID"] != "", cols].drop_duplicates().copy()
    frame.columns = ["team_id", "city", "nickname", "abbreviation"]
    team_frames.append(frame)

teams = pd.concat(team_frames).drop_duplicates().sort_values("team_id").reset_index(drop=True)
teams.shape, teams.head(40)

## Player Identity

Player identity also appears in participant slots. ID `0` means no real player in that slot.

In [ ]:
player_cols = [col for col in pbp.columns if "PLAYER" in col]
player_cols

In [ ]:
player_frames = []
for slot in ["PLAYER1", "PLAYER2", "PLAYER3"]:
    cols = [f"{slot}_ID", f"{slot}_NAME", f"{slot}_TEAM_ID", f"{slot}_TEAM_ABBREVIATION"]
    frame = pbp.loc[pbp[f"{slot}_ID"] != "0", cols].drop_duplicates().copy()
    frame.columns = ["player_id", "player_name", "team_id", "team_abbreviation"]
    player_frames.append(frame)

players_seen = pd.concat(player_frames).drop_duplicates().sort_values("player_id").reset_index(drop=True)
players_seen.shape, players_seen.head(25)

## Sparse Columns

Many columns are intentionally sparse because each event has only the fields relevant to that event.

In [ ]:
blank_counts = (pbp == "").sum().sort_values(ascending=False)
blank_report = pd.DataFrame({
    "blank_count": blank_counts,
    "blank_pct": (blank_counts / len(pbp) * 100).round(2),
})
blank_report.head(20)

## Event Types And Scores

`EVENTMSGTYPE` is the high-level event kind. `SCORE` and `SCOREMARGIN` appear only on score-bearing rows and period-end rows.

In [ ]:
pbp["EVENTMSGTYPE"].value_counts().sort_index()

In [ ]:
score_rows = pbp.loc[pbp["SCORE"] != "", ["GAME_ID", "EVENTNUM", "EVENTMSGTYPE", "PERIOD", "PCTIMESTRING", "SCORE", "SCOREMARGIN", "HOMEDESCRIPTION", "VISITORDESCRIPTION", "NEUTRALDESCRIPTION"]]
score_rows.head(20)

## One Game Example

Looking at one game makes the event shape obvious.

In [ ]:
first_game_id = pbp["GAME_ID"].iloc[0]
first_game = pbp[pbp["GAME_ID"] == first_game_id]
first_game_id, first_game.shape

In [ ]:
first_game[["EVENTNUM", "EVENTMSGTYPE", "PERIOD", "WCTIMESTRING", "PCTIMESTRING", "HOMEDESCRIPTION", "VISITORDESCRIPTION", "NEUTRALDESCRIPTION", "SCORE", "SCOREMARGIN"]].head(30)

In [ ]:
first_game[["EVENTNUM", "EVENTMSGTYPE", "PERIOD", "WCTIMESTRING", "PCTIMESTRING", "HOMEDESCRIPTION", "VISITORDESCRIPTION", "NEUTRALDESCRIPTION", "SCORE", "SCOREMARGIN"]].tail(10)

## M1 Schema Takeaway

Do not load this file into `player_game_logs`.

For M1:

- `player_game_logs` should come from `nba_api` box-score/player-log endpoints.
- shufinskiy should feed a raw `play_by_play_events` table if loaded now.
- The likely raw play-by-play natural key is `(GAME_ID, EVENTNUM)`.
- Every fact table still needs `event_time` and `ingested_at` so backtests can filter by point-in-time availability.